In [0]:
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from pyspark.sql.functions import lit, current_timestamp


# ============================================================
# 1. CHAMPION LOADER (finds and loads the champion model)
# ============================================================

class ChampionLoader:

    def __init__(self, model_name, alias="champion"):
        self.model_name = model_name
        self.alias = alias
        self.client = MlflowClient()

    def get_version(self):
        champion = self.client.get_model_version_by_alias(self.model_name, self.alias)
        return champion.version

    def load_model(self):
        model_uri = "models:/" + self.model_name + "@" + self.alias
        return mlflow.sklearn.load_model(model_uri)


# ============================================================
# 2. PREDICTOR (loads test data, drops non-feature columns, predicts)
# ============================================================

class Predictor:

    def __init__(self, spark, model, test_table, label_column, exclude_columns, id_columns):
        self.spark = spark
        self.model = model
        self.test_table = test_table
        self.label_column = label_column
        self.exclude_columns = exclude_columns   # columns dropped before predicting
        self.id_columns = id_columns             # columns kept only for reference in the output

    def load_test_data(self):
        # Full encoded test table: all columns, including label, product, city, etc.
        return self.spark.table(self.test_table).toPandas()

    def get_feature_columns(self, test_pandas_df):
        # Drop the label and the excluded columns -> what remains is X
        feature_columns = [
            c for c in test_pandas_df.columns
            if c != self.label_column and c not in self.exclude_columns
        ]

        # Check that X matches what the model was trained on
        trained_columns = list(self.model.feature_names_in_)
        missing = set(trained_columns) - set(feature_columns)
        extra = set(feature_columns) - set(trained_columns)

        if missing or extra:
            raise ValueError(
                "Test columns do not match training columns. "
                "Missing: " + str(missing) + " | Extra: " + str(extra)
            )

        # Return in the same order the model was trained with
        return trained_columns

    def predict(self):
        test_pandas_df = self.load_test_data()
        feature_columns = self.get_feature_columns(test_pandas_df)

        # Decimal -> float (same fix as in training)
        for c in feature_columns + [self.label_column]:
            test_pandas_df[c] = test_pandas_df[c].astype(float)

        X_test = test_pandas_df[feature_columns]

        result_df = test_pandas_df[self.id_columns].copy()
        result_df["actual_demand"] = test_pandas_df[self.label_column]
        result_df["predicted_demand"] = self.model.predict(X_test)
        result_df["absolute_error"] = (result_df["actual_demand"] - result_df["predicted_demand"]).abs()

        return result_df


# ============================================================
# 3. PREDICTION WRITER (saves predictions to Unity Catalog)
# ============================================================

class PredictionWriter:

    def __init__(self, spark, output_table):
        self.spark = spark
        self.output_table = output_table

    def write(self, result_pandas_df, model_name, model_version):
        result_spark_df = self.spark.createDataFrame(result_pandas_df)

        # Record which model produced these predictions, and when
        result_spark_df = (
            result_spark_df
            .withColumn("model_name", lit(model_name))
            .withColumn("model_version", lit(str(model_version)))
            .withColumn("prediction_ts", current_timestamp())
        )

        result_spark_df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(self.output_table)

        print("Predictions saved to: " + self.output_table)


# ============================================================
# 4. CONFIGURATION
# ============================================================

model_name = "oil_gas_mlops.ml_models.supply_chain_demand_model_sklearn"
test_table = "oil_gas_mlops.ml_data.test_data"
output_table = "oil_gas_mlops.ml_data.test_predictions"

label_column = "total_demand_quantity"

# Same list as training: these columns are dropped before predicting
exclude_columns = [
    "transaction_date",
    "product_name",
    "destination_city",
    "product_name_index",
    "destination_city_index"
]

# Kept only in the output table so each prediction can be traced to a row
id_columns = ["transaction_date", "product_name", "destination_city"]


# ============================================================
# 5. RUN
# ============================================================

mlflow.set_registry_uri("databricks-uc")

# Step 1: Load the champion model
loader = ChampionLoader(model_name=model_name)
model_version = loader.get_version()
model = loader.load_model()
print("Loaded champion: version " + str(model_version))

# Step 2: Load test data, drop non-feature columns, predict
predictor = Predictor(
    spark=spark,
    model=model,
    test_table=test_table,
    label_column=label_column,
    exclude_columns=exclude_columns,
    id_columns=id_columns
)
results_df = predictor.predict()

# Step 3: Save predictions to Unity Catalog
writer = PredictionWriter(spark=spark, output_table=output_table)
writer.write(results_df, model_name=model_name, model_version=model_version)

display(spark.table(output_table))